In [ ]:
# because the HP model uses a modified or older version of ZEN-garden the CSV output files must be imported from the main branch
#specific for HP (write values to demand_yearly_variation.csv)
import pandas as pd
import shutil

from pathlib import Path
import json

# CONFIGURATION
dataset_name = "energy_transition_example"

# Base paths
base_path = Path(r".\outputs")
output_dir = Path("./CSV_output") / dataset_name
esm_capacity_path = output_dir / "capacity_addition_aggregated_by_location.csv"

original_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP")
new_dir = Path(r"C:\Users\nicol\OneDrive\Dokumente\ZEN-garden_SP\Data_HP\ZEN-Model_HP_new")
original_dyv_path = original_dir / "set_carriers" / "HP" / "demand_yearly_variation.csv"
target_dyv_path = new_dir / "set_carriers" / "HP" / "demand_yearly_variation.csv"

# --- Step 1: Clone the original directory ---
if new_dir.exists():
    print(f"Directory already exists: {new_dir}")
else:
    shutil.copytree(original_dir, new_dir)
    print(f"Created clone of wind model input: {new_dir}")

# --- Step 2: Load year labels from system.json ---
system_path = base_path / dataset_name / "system.json"

print(f"Trying to open: {system_path}")
print(f"Does it exist? {system_path.exists()}")

with open(system_path, "r") as f:
    system_config = json.load(f)

reference_year = system_config["reference_year"]
interval = system_config["interval_between_years"]
optimized_years = system_config["optimized_years"]
year_labels = [str(reference_year + i * interval) for i in range(optimized_years)]


# --- Step 3: Load original demand_yearly_variation to preserve all nodes ---
df_existing = pd.read_csv(original_dyv_path)

# --- Step 4: Prepare updated wind data from ESM ---
country_map = {
    "DE": "DEU",
    "AT": "AUT",
    "IT": "ITA",
    "CZ": "CZE",
    "ROE": "ROE",
}

df = pd.read_csv(esm_capacity_path)
df_photovoltaics = df[df["technology"] == "heat_pump"]
df_grouped = df_photovoltaics.groupby(["location"]).sum(numeric_only=True).reset_index()
df_grouped["node"] = df_grouped["location"].map(country_map)
df_grouped = df_grouped[df_grouped["node"].notna()].drop(columns=["location"])

# Convert from GW to kW
df_grouped = df_grouped.copy()
for col in df_grouped.columns:
    if col not in ["node"]:  # Don't modify non-numeric columns
        df_grouped[col] = df_grouped[col] * 1_000_000

# Transpose to match demand_yearly_variation structure
df_new = df_grouped.set_index("node").T
df_new.index.name = "year"
df_new = df_new.reset_index()

# Fix year column using actual labels from system.json
df_new["year"] = year_labels[:len(df_new)]
df_new["year"] = df_new["year"].astype(str)
df_existing["year"] = df_existing["year"].astype(str)

# Set year as index for merging
df_new = df_new.set_index("year")
df_existing = df_existing.set_index("year")

# --- Step 5: Merge values only for matching columns ---
updated_cols = [col for col in df_new.columns if col in df_existing.columns]

for col in updated_cols:
    df_existing[col] = df_new[col].combine_first(df_existing[col])

# --- Step 6: Save final file ---
df_result = df_existing.reset_index()
df_result.to_csv(target_dyv_path, index=False)
print(f"Final demand_yearly_variation.csv written to:\n{target_dyv_path}")